In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [4]:
X = pd.read_parquet("../data/processed/luad_X.parquet")
labels = pd.read_csv("../data/processed/luad_labels.csv")

y = labels["label"]

X.shape, labels.shape

((574, 20530), (574, 3))

### Retrain to get learned coefficients

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
model = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    random_state=42,
)

model.fit(X_train_scaled, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",5000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

### Extract gene coeffecients

In [ ]:
# coef is the weight of a gene
coefs = model.coef_[0]

importance = pd.DataFrame({
    "gene": X.columns,
    "coef": coefs,
    "abs_coef": np.abs(coefs),
})

importance.head()

,gene,coef,abs_coef
0,ARHGEF10L,0.001692,0.001692
1,HIF3A,0.003501,0.003501
2,RNF17,-0.004475,0.004475
3,RNF10,0.002638,0.002638
4,RNF11,-0.004821,0.004821


In [9]:
top_tumor = importance.sort_values("coef", ascending=False).head(20)
top_tumor

,gene,coef,abs_coef
14760,DEFB127,0.034193,0.034193
17964,FKSG73,0.034193,0.034193
15197,SNORA38,0.022417,0.022417
2404,C7orf34,0.021256,0.021256
11789,SLC22A24,0.019950,0.019950
3355,HNRNPCL1,0.019788,0.019788
12758,CCL16,0.019760,0.019760
3591,CDX4,0.019644,0.019644
15327,C22orf42,0.018269,0.018269
3944,KBTBD13,0.018178,0.018178


In [10]:
top_normal = importance.sort_values("coef", ascending=True).head(20)
top_normal

,gene,coef,abs_coef
9759,SCARNA23,-0.032378,0.032378
15841,LOC643923,-0.022097,0.022097
10398,CSN3,-0.021872,0.021872
3403,IL13,-0.020622,0.020622
12303,EPB42,-0.020221,0.020221
11292,FAM71A,-0.018919,0.018919
11102,BARHL1,-0.018595,0.018595
16721,C20orf201,-0.017820,0.017820
7970,OR6K3,-0.017556,0.017556
6946,ZFATAS,-0.017093,0.017093


In [11]:
importance.sort_values("abs_coef", ascending=False).to_csv(
    "../data/processed/logistic_gene_importance.csv",
    index=False,
)

### Compare model coefficients with tumor-vs-normal expression difference

In [14]:
X_labeled = X.copy()
X_labeled["sample_type"] = labels["sample_type"].values
X_labeled["label"] = labels["label"].values

X_labeled[["sample_type", "label"]].head()

gene,sample_type,label
TCGA-69-7978-01,tumor,1.0
TCGA-62-8399-01,tumor,1.0
TCGA-78-7539-01,tumor,1.0
TCGA-50-5931-11,normal,0.0
TCGA-73-4658-01,tumor,1.0


In [15]:
# compute mean expression in tumor and normal

gene_cols = X.columns

tumor_mean = X_labeled[X_labeled["sample_type"] == "tumor"][gene_cols].mean()
normal_mean = X_labeled[X_labeled["sample_type"] == "normal"][gene_cols].mean()

In [16]:
# Compute difference in gene expression in tumor vs normal

diff = tumor_mean - normal_mean

In [17]:
gene_report = importance.copy()

gene_report["tumor_mean"] = gene_report["gene"].map(tumor_mean)
gene_report["normal_mean"] = gene_report["gene"].map(normal_mean)
gene_report["tumor_minus_normal"] = gene_report["gene"].map(diff)

gene_report.head()

,gene,coef,abs_coef,tumor_mean,normal_mean,tumor_minus_normal
0,ARHGEF10L,0.001692,0.001692,9.540719,9.517868,0.022851
1,HIF3A,0.003501,0.003501,6.877338,9.041775,-2.164437
2,RNF17,-0.004475,0.004475,0.551699,0.471249,0.080450
3,RNF10,0.002638,0.002638,11.770312,11.738858,0.031455
4,RNF11,-0.004821,0.004821,10.895561,11.507475,-0.611914


In [23]:
gene_report.sort_values("abs_coef", ascending=False).head(20)[
    ["gene", "coef", "tumor_mean", "normal_mean", "tumor_minus_normal"]
]

,gene,coef,tumor_mean,normal_mean,tumor_minus_normal
17964,FKSG73,0.034193,0.005812,0.000000,0.005812
14760,DEFB127,0.034193,0.001052,0.016125,-0.015073
9759,SCARNA23,-0.032378,0.001128,0.011892,-0.010764
15197,SNORA38,0.022417,0.005109,0.000000,0.005109
15841,LOC643923,-0.022097,0.065920,0.236015,-0.170095
10398,CSN3,-0.021872,0.042772,0.061217,-0.018445
2404,C7orf34,0.021256,0.257439,0.228693,0.028746
3403,IL13,-0.020622,0.780545,1.601625,-0.821080
12303,EPB42,-0.020221,0.157139,1.466269,-1.309130
11789,SLC22A24,0.019950,0.013990,0.011334,0.002656


In [18]:
gene_report.sort_values("coef", ascending=False).head(20)

,gene,coef,abs_coef,tumor_mean,normal_mean,tumor_minus_normal
14760,DEFB127,0.034193,0.034193,0.001052,0.016125,-0.015073
17964,FKSG73,0.034193,0.034193,0.005812,0.000000,0.005812
15197,SNORA38,0.022417,0.022417,0.005109,0.000000,0.005109
2404,C7orf34,0.021256,0.021256,0.257439,0.228693,0.028746
11789,SLC22A24,0.019950,0.019950,0.013990,0.011334,0.002656
3355,HNRNPCL1,0.019788,0.019788,0.041830,0.000000,0.041830
12758,CCL16,0.019760,0.019760,0.808234,1.584759,-0.776526
3591,CDX4,0.019644,0.019644,0.001002,0.000000,0.001002
15327,C22orf42,0.018269,0.018269,0.025644,0.033834,-0.008190
3944,KBTBD13,0.018178,0.018178,0.102786,0.186708,-0.083922


In [19]:
gene_report.sort_values("coef", ascending=True).head(20)

,gene,coef,abs_coef,tumor_mean,normal_mean,tumor_minus_normal
9759,SCARNA23,-0.032378,0.032378,0.001128,0.011892,-0.010764
15841,LOC643923,-0.022097,0.022097,0.065920,0.236015,-0.170095
10398,CSN3,-0.021872,0.021872,0.042772,0.061217,-0.018445
3403,IL13,-0.020622,0.020622,0.780545,1.601625,-0.821080
12303,EPB42,-0.020221,0.020221,0.157139,1.466269,-1.309130
11292,FAM71A,-0.018919,0.018919,0.381267,1.997300,-1.616033
11102,BARHL1,-0.018595,0.018595,0.075210,0.114193,-0.038983
16721,C20orf201,-0.017820,0.017820,0.930721,1.800334,-0.869613
7970,OR6K3,-0.017556,0.017556,0.188966,1.937473,-1.748507
6946,ZFATAS,-0.017093,0.017093,0.178941,0.218149,-0.039209


In [20]:
top_up_in_tumor = gene_report.sort_values(
    "tumor_minus_normal",
    ascending=False,
).head(20)

top_up_in_normal = gene_report.sort_values(
    "tumor_minus_normal",
    ascending=True,
).head(20)

In [21]:
top_up_in_tumor[["gene", "coef", "tumor_mean", "normal_mean", "tumor_minus_normal"]]

,gene,coef,tumor_mean,normal_mean,tumor_minus_normal
17892,FAM83A,0.012736,10.108718,3.877571,6.231147
4238,COL11A1,0.004480,8.157597,2.305314,5.852283
19773,CST1,0.002443,6.984317,1.136517,5.847800
3948,LOC84740,0.014810,9.063464,3.837763,5.225702
3296,CYP24A1,0.010676,8.355195,3.144085,5.211110
12960,MMP11,0.008202,10.158472,5.003664,5.154808
7271,ABCA12,0.010460,6.413655,1.416124,4.997531
8074,CA9,0.006049,6.478163,1.553093,4.925069
12962,MMP13,0.004963,6.702057,1.821312,4.880745
12041,PPP1R14D,0.007169,5.952637,1.149886,4.802750


In [22]:
top_up_in_normal[["gene", "coef", "tumor_mean", "normal_mean", "tumor_minus_normal"]]

,gene,coef,tumor_mean,normal_mean,tumor_minus_normal
19926,SFTPC,0.001357,10.406971,18.965454,-8.558483
8612,SLC6A4,-0.001695,2.955707,10.926241,-7.970534
19134,CLDN18,0.000161,7.404636,14.457229,-7.052593
8420,AGER,-0.002465,8.652601,15.411381,-6.758781
6791,ITLN2,-0.001741,2.222745,8.980227,-6.757482
3177,LGI3,0.002372,4.206939,10.844725,-6.637786
4524,C13orf36,-0.006967,1.828323,8.291746,-6.463423
9539,FABP4,-0.000530,5.167863,11.330927,-6.163064
11461,HBA1,-0.001570,5.565666,11.552702,-5.987035
14114,GPM6A,0.000040,4.249797,10.215029,-5.965232
